# Langfuse 프롬프트 관리 

---

## 환경 설정 및 준비

### (1) Env 환경변수

```markdown
LANGFUSE_SECRET_KEY=sk-
LANGFUSE_PUBLIC_KEY=pk-
# 🇪🇺 EU region
LANGFUSE_HOST=https://cloud.langfuse.com
```


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### (2) 기본 라이브러리

In [2]:
import os
from glob import glob
from pprint import pprint
import json
import warnings
warnings.filterwarnings("ignore")

### (3) Langfuse 콜백 핸들러 설정

In [38]:
from langfuse.langchain import CallbackHandler 

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

### (4) Langfuse 클라이언트 설정

In [3]:
from langfuse import get_client

# Langfuse 클라이언트 초기화
langfuse = get_client()

# 연결 테스트
assert langfuse.auth_check()

---

## 프롬프트 관리 개요

Langfuse는 **프롬프트 CMS(Content Management System)** 기능을 제공

- **버전 관리**: 프롬프트의 모든 변경사항을 추적하고 롤백 가능
- **협업**: 팀원들과 함께 프롬프트를 편집하고 관리
- **배포 관리**: 라벨을 통해 코드 변경 없이 환경별 배포
- **성능 모니터링**: 프롬프트 버전별 성능 메트릭 비교
- **실시간 테스트**: 플레이그라운드에서 즉시 테스트 가능

---

## 1. 프롬프트 생성

### 1.1 텍스트 프롬프트 생성

In [4]:
# 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic",  # 프롬프트 이름
    type="text",          
    prompt="{{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?",
    labels=["production"],           # 프로덕션 레이블
    tags=["movie", "qa", "text"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

### 1.2 챗 프롬프트 생성

In [5]:
# 챗 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic-chat",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

### 1.3 메시지 플레이스홀더가 있는 챗 프롬프트

In [48]:
# 메시지 플레이스홀더를 포함한 챗 프롬프트
langfuse.create_prompt(
    name="movie-critic-with-history",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "type": "placeholder",
            "name": "chat_history"  # 대화 히스토리 삽입 지점
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대해 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],
    tags=["movie", "qa", "chat", "history"]
)

### **[실습 1]**
텍스트 기반 프롬프트와 chat 기반 프롬프트를 각각 구현하고, Langfuse UI에서 확인하세요.

In [7]:
# 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="service-exam-text-prompt",  # 프롬프트 이름
    type="text",          
    prompt="{{serviceLevel}} 서비스 운영자로서, {{quality}}를 어떻게 생각하시나요?",
    labels=["production"],           # 프로덕션 레이블
    tags=["service", "qa", "text"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

<details>
<summary>💡 정답 보기</summary>

```python
# 간단한 문서 요약용 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="document-summarizer",
    type="text",
    prompt="""다음 {{document_type}}를 {{summary_length}} 단어 이내로 요약해주세요:

제목: {{document_title}}

내용:
{{document_content}}

{{summary_style}} 형태로 요약해주세요.""",
    labels=["production"],
    tags=["document", "summary", "simple", "text"],
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.2,
        "max_tokens": 800
    }
)
```
</details>


In [8]:
# 챗 프롬프트 생성
langfuse.create_prompt(
    name="service-exam-chat-prompt",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{serviceLevel}} 서비스 운영자입니다."
        },
        {
            "role": "user",
            "content": "서비스 품질을 {{qualrity}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["service", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

<details>
<summary>💡 정답 보기</summary>

```python
# 간단한 문서 요약용 챗 프롬프트 생성
langfuse.create_prompt(
    name="document-summarizer-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 문서 요약 전문가입니다. {{summary_style}} 형태로 {{summary_length}} 단어 이내로 요약해주세요."
        },
        {
            "role": "user",
            "content": """다음 {{document_type}}를 요약해주세요:

제목: {{document_title}}

내용:
{{document_content}}"""
        }
    ],
    labels=["production"],
    tags=["document", "summary", "chat", "simple"],
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.2,
        "max_tokens": 800
    }
)
```
</details>


---

## 2. 프롬프트 활용

### 2.1 기본 프롬프트 가져오기

In [47]:
# 프로덕션 버전 가져오기
prompt = langfuse.get_prompt("movie-critic")

# 프롬프트 정보 출력
print(f"모델: {prompt.config['model']}")
print(f"온도: {prompt.config['temperature']}")
print(f"라벨: {prompt.labels}")
print(f"태그: {prompt.tags}")
print(f"프롬프트: {prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(prompt.get_langchain_prompt())

모델: gpt-4.1
온도: 0.7
라벨: ['production', 'latest']
태그: ['movie', 'qa', 'text', 'detailed']
프롬프트: 당신은 {{criticLevel}} 영화 평론가입니다.

영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.
----------------------------------------------------------------------------------------------------
당신은 {criticLevel} 영화 평론가입니다.

영화 {movie}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.


### 2.2 compile 메서드 사용

- compile 메서드로 변수 삽입

In [10]:
# compile 메서드로 변수 삽입
compiled_prompt = prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_prompt)

전문가 영화 평론가로서, 인셉션를 어떻게 생각하시나요?


### 2.3 챗 프롬프트 가져오기 및 컴파일

In [11]:
# 챗 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 챗 프롬프트 정보 출력
print(f"모델: {chat_prompt.config['model']}")
print(f"온도: {chat_prompt.config['temperature']}")
print(f"라벨: {chat_prompt.labels}")
print(f"태그: {chat_prompt.tags}")
print(f"프롬프트: {chat_prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(chat_prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production', 'latest']
태그: ['movie', 'qa', 'chat']
프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
[('system', '당신은 {criticLevel} 영화 평론가입니다.'), ('user', '영화 {movie}를 어떻게 생각하시나요?')]


In [12]:
# 챗 프롬프트 컴파일
compiled_chat_prompt = chat_prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_chat_prompt)

[{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}, {'role': 'user', 'content': '영화 인셉션를 어떻게 생각하시나요?'}]


### 2.4 메시지 플레이스홀더 활용

In [13]:
# 플레이스홀더가 있는 챗 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_with_history = prompt_with_history.compile(
            criticLevel="전문가",
            movie="인셉션", 
            chat_history=chat_history
        )

for message in compiled_with_history:
    print(message)
    print("-" * 20)

{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}
--------------------
{'role': 'user', 'content': '안녕하세요!'}
--------------------
{'role': 'assistant', 'content': '안녕하세요! 영화에 대해 이야기해볼까요?'}
--------------------
{'role': 'user', 'content': '영화 인셉션에 대해 어떻게 생각하시나요?'}
--------------------


### **[실습 2]**
"movie-critic-chat" 프롬프트를 Langfuse에서 가져와서 내용을 출력하고, compile 메서드를 사용해 변수에 적절한 값을 추가해보세요.

In [22]:
# chat 프롬프트 가져오기 및 컴파일

# 챗 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 랭체인 호환 프롬프트 출력
for message in compiled_chat_prompt:
    print(message)
print("-" * 80)

compiled_chat_prompt = chat_prompt.compile(criticLevel="초보자", movie="케데헌")

for message in compiled_chat_prompt:
    print(message)

{'role': 'system', 'content': '당신은 초보자 영화 평론가입니다.'}
{'role': 'user', 'content': '영화 케데헌를 어떻게 생각하시나요?'}
--------------------------------------------------------------------------------
{'role': 'system', 'content': '당신은 초보자 영화 평론가입니다.'}
{'role': 'user', 'content': '영화 케데헌를 어떻게 생각하시나요?'}


<details>
<summary>💡 정답 보기</summary>

```python
# chat 프롬프트 가져오기 및 컴파일
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_chat_prompt = chat_prompt.compile(
    criticLevel="전문가", 
    movie="인셉션"
)

for message in compiled_chat_prompt:
    print(message)
    print("-" * 20)
```
</details>


---

## 3. 프롬프트 버전 관리

### 3.1 새로운 버전 생성

In [23]:
# 새로운 버전 생성 (같은 이름 사용)
langfuse.create_prompt(
    name="movie-critic",  # 같은 이름 사용
    type="text",          
    prompt="당신은 {{criticLevel}} 영화 평론가입니다.\n\n영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text", "detailed"],    # 태그 업데이트
    config={
        "model": "gpt-4.1",  # 모델 업그레이드
        "temperature": 0.7,
        "max_tokens": 1000  # 토큰 수 증가
    }
)

### 3.2 특정 버전 가져오기

In [24]:
# 특정 버전 가져오기
prompt_v1 = langfuse.get_prompt("movie-critic", version=1)
prompt_v2 = langfuse.get_prompt("movie-critic", version=2)

# 버전별 비교
print(f"V1 프롬프트: {prompt_v1.prompt}")
print(f"V2 프롬프트: {prompt_v2.prompt}")
print(f"V1 모델: {prompt_v1.config['model']}")
print(f"V2 모델: {prompt_v2.config['model']}")

V1 프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
V2 프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
V1 모델: gpt-4.1-mini
V2 모델: gpt-4.1-mini


### 3.3 라벨 관리

In [25]:
# 특정 라벨로 프롬프트 생성 (같은 이름을 사용하면 새로운 버전으로 생성됨)
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# 라벨별 프롬프트 가져오기
prompt_production = langfuse.get_prompt("movie-critic-chat", label="production")
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="staging")
prompt_latest = langfuse.get_prompt("movie-critic-chat", label="latest")

In [26]:
# 라벨별 프롬프트 출력
print(f"Production 프롬프트: {prompt_production.prompt}")
print("-" * 100)
print(f"Staging 프롬프트: {prompt_staging.prompt}")
print("-" * 100)
print(f"Latest 프롬프트: {prompt_latest.prompt}")

Production 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
Staging 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]
----------------------------------------------------------------------------------------------------
Latest 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]


### 3.4 라벨 업데이트

In [36]:
# 기존 프롬프트 버전의 라벨 업데이트
langfuse.update_prompt(
    name="movie-critic-chat",
    version=3,
    new_labels=["production", "v2-stable"]
)

Prompt_Chat(prompt=[ChatMessageWithPlaceholders_Chatmessage(role='system', content='당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.', type='chatmessage'), ChatMessageWithPlaceholders_Chatmessage(role='user', content='영화 {{movie}}에 대한 평론을 작성해주세요.', type='chatmessage')], name='movie-critic-chat', version=3, config={}, labels=['production', 'v2-stable', 'staging', 'latest'], tags=['movie', 'qa', 'chat', 'detailed'], commit_message=None, resolution_graph=None, type='chat', id='ba9acaa3-1424-40d1-9e3f-f8c033952374', updatedAt='2025-09-23T01:16:28.862Z', isActive=None, createdBy='API', createdAt='2025-09-23T00:54:47.310Z', projectId='cmfvsy4qh029aad083kawrsv5')

### **[실습 3]**
"movie-critic-chat" 프롬프트를 수정하고, labels 속성은 "staging"으로 지정한 후, staging 버전을 가져와서 내용을 출력하세요.

In [34]:
# staging 라벨 생성
langfuse.update_prompt(
    name="movie-critic-chat",
    version=3,
    new_labels=["staging"]
)
prompt_production = langfuse.get_prompt("movie-critic-chat", label="staging")

for prompt in prompt_production.prompt:
    print(prompt)

# staging 라벨 가져오기
# 여기에 코드를 작성하세요

{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}
{'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}


<details>
<summary>💡 정답 보기</summary>

```python
# staging 라벨 생성
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 일반적이고 상세하며 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# staging 라벨 가져오기
prompt_staging = langfuse.get_prompt(
    name="movie-critic-chat",
    label="staging"
)

# 라벨별 프롬프트 출력
print(f"Staging 프롬프트: {prompt_staging.prompt}")
```
</details>


---

## 4. LangChain과의 통합

### 4.1 텍스트 프롬프트와 LangChain 통합

In [40]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Langfuse 프롬프트를 LangChain과 통합
prompt = langfuse.get_prompt("movie-critic", label="production")

langchain_prompt = PromptTemplate.from_template(
    prompt.get_langchain_prompt()
)
langchain_prompt.metadata = {"langfuse_prompt": prompt}   # Langfuse 자동 링크를 위한 메타데이터

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7),
    max_completion_tokens=prompt.config.get("max_tokens", 500)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
)

print(response.content)

물론입니다. 영화 **〈인셉션(Inception, 2010)〉**은 크리스토퍼 놀란 감독이 연출하고, 레오나르도 디카프리오, 조셉 고든 레빗, 마리옹 꼬띠아르, 톰 하디 등이 출연한 SF 스릴러 영화입니다. 이 작품은 꿈과 현실, 무의식의 경계를 넘나드는 독창적인 세계관과 정교한 플롯, 뛰어난 시각 효과로 극찬을 받았습니다. 아래에 연출, 연기, 스토리, 시각적 효과 측면에서 상세히 분석하겠습니다.

---

## 1. 연출

크리스토퍼 놀란 감독의 연출은 <인셉션>의 가장 큰 강점 중 하나입니다. 그는 현실과 꿈의 경계를 모호하게 만들면서도 관객이 혼란에 빠지지 않도록 치밀하게 구조를 설계했습니다. 꿈의 각 레이어가 시간의 흐름, 공간의 왜곡, 중력의 변화 등 다양한 규칙을 갖고 있다는 점을 시각적으로 명확하게 전달했으며, 특히 꿈 속에서의 액션 신이나 파리의 거리 접기, 호텔 복도의 무중력 격투 등은 감각적인 연출력의 결정체입니다.

놀란은 복잡한 플롯을 여러 층위로 쌓아올리면서도 각 장면의 리듬과 긴장감을 잃지 않고, 관객이 몰입할 수 있도록 세심하게 통제합니다. 음악감독 한스 짐머와의 협업으로 만들어진 사운드트랙 역시 영화의 몰입도를 극대화합니다.

## 2. 연기

레오나르도 디카프리오는 주인공 ‘돔 코브’ 역을 맡아 깊은 내면 연기와 감정의 진폭을 섬세하게 표현합니다. 죄책감과 상실, 집착 사이에서 흔들리는 인물을 설득력 있게 그려냈습니다. 조셉 고든 레빗(아서 역)은 침착하고 이성적인 조력자로서, 톰 하디(임스 역)는 유쾌함과 카리스마를 동시에 보여주며 극에 활력을 불어넣습니다.

특히 마리옹 꼬띠아르는 코브의 아내 ‘멀’로 등장해, 망령처럼 주인공을 따라다니는 존재로서 불안과 슬픔, 위험함을 동시에 발산하며 인상적인 연기를 선보입니다. 각 배우는 자신의 캐릭터에 깊이 몰입해 영화의 복잡한 세계관 안에서 현실감을 부여합니다.

## 3. 스토리

<인셉션>의 스토리는 ‘타인의 꿈에 들어가 생각을 심는다’는 독특한 설정에서 출발합니다. 이 ‘인셉션’

### 4.2 챗 프롬프트와 LangChain 통합

In [39]:
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 통합
chat_prompt = langfuse.get_prompt("movie-critic-chat", label="production", type="chat")

langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 체인 실행
chain = langchain_chat_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}
)

print(response.content)

크리스토퍼 놀란 감독의 2010년작 《인셉션》(Inception)은 현대 SF 스릴러의 경계를 새롭게 정의한 영화다. 영화는 꿈과 현실, 무의식과 의식의 경계를 넘나드는 독창적인 플롯과 정교한 구조, 그리고 철학적 질문을 통해 관객에게 깊은 사유와 긴장감을 동시에 선사한다.

**서사 구조와 내러티브의 미학**

《인셉션》의 서사는 ‘꿈속의 꿈’이라는 다층적 내러티브 구조 위에 세워져 있다. 도미닉 콥(레오나르도 디카프리오)과 그의 팀이 타인의 무의식에 침투해 아이디어를 심는 ‘인셉션’ 작전을 수행하는 과정은, 일반적인 하이스트 무비의 전형적인 구조를 차용하면서도, 시간과 공간의 개념을 재해석한다. 꿈의 레벨이 깊어질수록 시간의 흐름이 달라지고, 현실과 환상의 경계가 모호해진다. 이로 인해 관객은 끊임없이 ‘지금 보고 있는 것이 현실인가, 꿈인가’라는 의문을 품게 된다.

**주제와 철학적 탐구**

영화는 단순한 SF 액션의 외피를 두르고 있으나, 그 중심에는 ‘현실이란 무엇인가’, ‘기억과 죄책감이 인간에게 미치는 영향’이라는 심오한 주제가 자리잡고 있다. 콥의 트라우마와 죄책감, 아내 말(마리옹 코티야르)에 대한 집착은 영화 전반을 이끄는 심리적 동력이다. 이로써 《인셉션》은 꿈과 현실을 구분하는 물리적 장치(토템)를 통해 인간의 본질적 불안과 갈등, 그리고 구원의 가능성에 대해 질문을 던진다.

**연출과 시각적 완성도**

놀란의 연출은 이 영화에서 정점에 달한다. CG와 실제 촬영, 스턴트가 절묘하게 결합된 액션 시퀀스(예: 중력 없는 회전 복도 신)는 시각적 혁신 그 자체다. 한스 짐머의 음악 역시 꿈과 무의식의 몽환적 분위기를 극대화하며, 영화의 정서적 깊이를 더한다. 세트와 미술, 촬영 역시 현실과 환상의 경계를 시각적으로 표현하는 데 탁월하다.

**배우들의 연기와 캐릭터**

레오나르도 디카프리오를 비롯한 조셉 고든 레빗, 엘렌 페이지(현 엘리엇 페이지), 톰 하디 등 앙상블 캐스팅은 각기 인상적인 연기를 선보인다. 특히 콥의 내면적 고통을

### 4.3 플레이스홀더가 있는 프롬프트와 LangChain 통합

In [41]:
from langchain_core.prompts import MessagesPlaceholder

# 플레이스홀더가 있는 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# LangChain 호환 프롬프트로 변환 (미해결 플레이스홀더는 MessagesPlaceholder로 변환)
langchain_prompt_with_placeholder = ChatPromptTemplate.from_messages(
    prompt_with_history.get_langchain_prompt()
)
langchain_prompt_with_placeholder.metadata = {"langfuse_prompt": prompt_with_history}

# chain 생성
chain = langchain_prompt_with_placeholder | model

# 실행 시 플레이스홀더 값 제공
chat_history = [
    {"role": "user", "content": "안녕하세요! 영화 예산에 대해서 이야기해볼까요?"},
    {"role": "assistant", "content": "안녕하세요! 영화 예산에 대해 어떻게 도와드릴까요?"}
]

response = chain.invoke({
    "criticLevel": "전문가", 
    "movie": "인셉션",
    "chat_history": chat_history
}, config={"callbacks": [langfuse_handler]})  # Langfuse 트레이싱을 위한 콜백

print(response.content)

Placeholders ['chat_history'] have not been resolved. Pass them as keyword arguments to compile().


영화 **인셉션(Inception, 2010)**은 크리스토퍼 놀란 감독의 대표작 중 하나로, 현대 SF 영화의 새로운 지평을 열었다는 평가를 받는 작품입니다. 이 영화는 꿈과 현실, 무의식과 의식의 경계를 다루는 독특한 스토리텔링, 그리고 놀란 특유의 복잡한 구조와 정교한 연출로 많은 관객과 평론가들에게 깊은 인상을 남겼습니다.

**장점**
1. **창의적인 설정과 플롯**  
   꿈속의 꿈이라는 다층적 구조, '인셉션'이라는 참신한 아이디어는 관객들에게 독특한 영화적 경험을 제공합니다. 각 꿈의 레이어마다 다른 시간의 흐름과 스릴 넘치는 액션이 어우러지면서, 놀란 감독의 상상력이 극대화되었습니다.

2. **시각적 완성도**  
   1억 6천만 달러(약 1,600억 원)에 달하는 대규모 예산이 아깝지 않게, 시각효과와 미술, 촬영이 모두 뛰어납니다. 파리 거리가 접히는 장면이나 무중력 액션 시퀀스 등은 영화사에 남을 명장면입니다.

3. **연기와 캐릭터**  
   레오나르도 디카프리오를 비롯한 조셉 고든 레빗, 엘렌 페이지(현재 엘리엇 페이지), 톰 하디 등 출연진의 안정적인 연기도 몰입도를 높였습니다.

4. **음악**  
   한스 짐머의 음악은 영화의 분위기를 한층 고조시키며, ‘Time’ 트랙은 명곡으로 오랫동안 회자되고 있습니다.

**단점 및 아쉬운 점**
- 복잡한 구조와 다중 플롯 때문에 일부 관객에게는 난해하게 느껴질 수 있습니다.
- 감정적인 서사(코브와 말의 관계)에 공감하기 어려웠다는 평가도 있습니다.

**평론가적 총평**  
<인셉션>은 상업성과 예술성을 모두 잡은 드문 블록버스터입니다. 깊이 있는 주제의식과 압도적인 시각적 스펙터클, 그리고 긴장감 넘치는 전개가 어우러져, 2010년대를 대표하는 현대 영화로 자리잡았습니다. 단순한 액션 오락영화를 넘어 ‘꿈’이라는 인간 내면의 복잡한 세계를 영화적으로 해석한 명작이라고 말할 수 있습니다.

혹시 인셉션의 예산, 제작 비화, 혹은 특정 장면에 대해 더 궁금한 점이 있으신가

- 특정 라벨 가져오기

In [42]:
# 특정 라벨 가져오기
prompt_staging = langfuse.get_prompt("movie-critic", label="latest")  # production, latest

# 프롬프트 출력
print(f"모델: {prompt_staging.config['model']}")
print(f"온도: {prompt_staging.config['temperature']}")
print(f"라벨: {prompt_staging.labels}")
print(f"프롬프트: {prompt_staging.prompt}")

모델: gpt-4.1
온도: 0.7
라벨: ['production', 'latest']
프롬프트: 당신은 {{criticLevel}} 영화 평론가입니다.

영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.


In [43]:
from langchain_core.prompts import PromptTemplate

# Langchain과 통합 
langchain_prompt = PromptTemplate.from_template(
    prompt_staging.get_langchain_prompt(),
)
langchain_prompt.metadata = {"langfuse_prompt": prompt_staging}

print(langchain_prompt.format(criticLevel="비평가", movie="인셉션"))

당신은 비평가 영화 평론가입니다.

영화 인셉션에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.


In [44]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화 (프롬프트 설정에서 가져온 값 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # 콜백 핸들러 추가
)

# 응답 출력
print(response.content)

물론입니다. 크리스토퍼 놀란 감독의 **<인셉션(Inception, 2010)>**은 21세기 영화사에서 독보적인 위치를 차지하는 작품입니다. 이 영화는 꿈과 현실, 무의식과 의식의 경계를 넘나드는 독창적이고 복합적인 구조로 관객에게 강렬한 충격과 깊은 여운을 남겼습니다. 아래에서 연출, 연기, 스토리, 시각적 효과 측면에서 상세하게 분석하겠습니다.

---

### 1. 연출

크리스토퍼 놀란은 영화 전체를 치밀하게 설계하며, 복잡한 플롯과 다층적인 세계관을 탁월하게 통제합니다. 꿈속의 꿈이라는 구조는 자칫 혼란스러울 수 있으나, 놀란은 각 층의 시공간적 규칙을 명확하게 제시하고, 시계, 음악, 시각적 단서 등으로 관객이 따라올 수 있도록 배려합니다.  
특히, 클라이맥스에서 3~4단계의 꿈이 동시다발적으로 전개되는 시퀀스는 놀란 특유의 장인정신이 빛나는 부분입니다. 현실과 환상의 경계를 희미하게 하면서도, 극적인 긴장감을 놓치지 않는 연출력은 경이로울 정도입니다.

---

### 2. 연기

주연 레오나르도 디카프리오(돔 코브 역)는 죄책감과 상실, 집착이라는 복잡한 내면을 설득력 있게 표현하며, 영화의 감정적 중심을 단단히 지탱합니다. 마리옹 꼬띠아르(맬 역)는 치명적이고 매혹적인 존재감을 발산하며, 꿈과 현실을 교란하는 키 역할을 훌륭히 소화합니다.  
조셉 고든 레빗, 엘렌 페이지, 톰 하디, 킬리언 머피 등 조연진 역시 각자의 역할에 맞게 섬세한 연기를 선보입니다. 이들의 유기적 호흡이 복잡한 플롯 속에서 인물의 동기와 감정을 효과적으로 전달합니다.

---

### 3. 스토리

<인셉션>의 스토리는 단순한 SF 도둑 이야기에서 출발하지만, 인간의 무의식, 상실, 죄책감, 꿈과 현실의 경계 등 심오한 주제를 탐구합니다. "인셉션"이라는 개념 자체가 타인의 무의식 속에 생각을 심는다는 매우 철학적이고 독창적인 발상입니다.  
놀란은 이 복잡한 아이디어를 액션, 스릴러, 멜로, 심리극 등 다양한 장르적 요소와 결합시켜, 지적 쾌감과 감정적 몰입을 동

### **[실습 4]**
앞에서 정의한 텍스트 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.


In [46]:
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 통합
chat_prompt = langfuse.get_prompt("service-exam-chat-prompt", label="production", type="chat")

langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 체인 실행
chain = langchain_chat_prompt | model
response = chain.invoke(
    input={"serviceLevel": "한국초보자", "serviceName": "이커머스"},
    config={"callbacks": [langfuse_handler]}
)

print(response.content)

어떤 이커머스 서비스가 "가장 좋다"고 말하기는 쉽지 않습니다. 사용 목적, 판매 상품, 타겟 고객, 예산, 운영 방식 등 여러 요소에 따라 적합한 서비스가 다르기 때문입니다. 아래는 한국에서 많이 사용되는 대표적인 이커머스 플랫폼과 각 특징입니다.

1. 쿠팡  
- 특징: 빠른 배송(로켓배송), 많은 이용자, 모바일 사용 편리  
- 장점: 대형 마켓, 신뢰성, 광고/프로모션 다양  
- 단점: 입점 심사, 수수료, 경쟁 심함

2. 네이버 스마트스토어  
- 특징: 네이버 검색·쇼핑과 연동, 소규모 판매자도 쉽게 시작  
- 장점: 초기 진입 장벽 낮음, 네이버 페이 등 네이버 서비스 활용  
- 단점: 경쟁 치열, 상위 노출 어렵기도 함

3. 11번가, G마켓, 옥션 등  
- 특징: 오래된 오픈마켓, 다양한 상품군  
- 장점: 고객층 넓음, 마케팅/광고 툴 다양  
- 단점: 수수료, 경쟁 심함, UI/UX는 다소 올드할 수 있음

4. 카페24, 메이크샵, 고도몰 등(자사몰 구축 솔루션)  
- 특징: 브랜드몰 직접 구축  
- 장점: 브랜드 정체성 구축, 데이터 소유, 자유로운 마케팅  
- 단점: 초기 세팅 필요, 직접 운영·마케팅 필요

**결론:**  
- 빠르게 시작하고 싶거나, 소규모로 판매를 원하면 네이버 스마트스토어 추천  
- 대형 시장에서 빠르게 성장하고 싶으면 쿠팡, 11번가, G마켓 등 오픈마켓  
- 자체 브랜드로 장기적으로 키우고 싶으면 카페24 등 자사몰 구축

당신의 상황(예산, 경험, 판매 상품 등)에 따라 가장 좋은 서비스를 선택하는 것이 중요합니다. 추가 정보가 있으시면 더 맞춤형으로 추천드릴 수 있습니다!


<details>
<summary>💡 정답 보기</summary>

```python
# Langfuse 프롬프트를 LangChain과 통합
prompt = langfuse.get_prompt("document-summarizer", label="production")

langchain_prompt = PromptTemplate.from_template(
    prompt.get_langchain_prompt()
)

# Langfuse 자동 링크를 위한 메타데이터 설정
langchain_prompt.metadata = {"langfuse_prompt": prompt}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.2),
    max_completion_tokens=prompt.config.get("max_tokens", 800)
)

# 체인 생성 및 실행
chain = langchain_prompt | model

# 샘플 문서 내용
sample_document = """
Langfuse는 LLM 애플리케이션을 위한 오픈소스 관찰 가능성 플랫폼입니다. 
개발자들이 LLM 애플리케이션을 디버그, 분석, 반복 개발할 수 있도록 도와줍니다.

주요 기능:
- 상세한 추적 및 관찰 가능성
- 프롬프트 관리 및 버전 제어
- 평가 및 데이터셋 관리
- 대시보드를 통한 성능 모니터링
- 다양한 프레임워크와의 통합 (LangChain, OpenAI, LangGraph 등)

Langfuse를 사용하면 프로덕션 환경에서 LLM 애플리케이션의 품질과 성능을 
지속적으로 개선할 수 있습니다.
"""

# 문서 요약 실행 (Langfuse 트레이싱 포함)
response = chain.invoke(
    input={
        "document_type": "기술 문서",
        "summary_length": "100",
        "document_title": "Langfuse 소개",
        "document_content": sample_document,
        "summary_style": "불릿포인트"
    },
    config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
)

print("요약 결과:")
print(response.content)

# Langfuse로 모든 데이터 전송 완료
langfuse.flush()
```
</details>


### **[실습 5]**
앞에서 정의한 chat 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

In [ ]:
# 여기에 코드를 추가하세요.

<details>
<summary>💡 정답 보기</summary>

```python
# Langfuse 챗 프롬프트를 LangChain과 통합
chat_prompt = langfuse.get_prompt("document-summarizer-chat", type="chat")

# ChatPromptTemplate로 변환 (챗 메시지 형태)
langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)

# Langfuse 자동 링크를 위한 메타데이터 설정
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=chat_prompt.config.get("model", "gpt-4.1-mini"),
    temperature=chat_prompt.config.get("temperature", 0.2),
    max_completion_tokens=chat_prompt.config.get("max_tokens", 800)
)

# 체인 생성
chain = langchain_chat_prompt | model

# 샘플 문서 내용들
sample_documents = {
    "technical": """
Langfuse는 LLM 애플리케이션을 위한 오픈소스 관찰 가능성 플랫폼입니다. 
개발자들이 LLM 애플리케이션을 디버그, 분석, 반복 개발할 수 있도록 도와줍니다.

주요 기능:
- 상세한 추적 및 관찰 가능성: 모든 LLM 호출과 체인 실행을 추적
- 프롬프트 관리 및 버전 제어: 프롬프트의 버전 관리와 A/B 테스트
- 평가 및 데이터셋 관리: 모델 성능 평가와 벤치마크 데이터셋
- 대시보드를 통한 성능 모니터링: 실시간 성능 지표와 사용량 분석
- 다양한 프레임워크와의 통합: LangChain, OpenAI, LangGraph 등과 원활한 통합

Langfuse를 사용하면 프로덕션 환경에서 LLM 애플리케이션의 품질과 성능을 
지속적으로 개선할 수 있으며, 팀 간의 협업도 효율적으로 진행할 수 있습니다.
""",
    "meeting": """
2024년 3월 주간 개발팀 회의록

참석자: 김개발, 이기획, 박디자인, 최테스트
일시: 2024년 3월 15일 오후 2시

주요 안건:
1. 프로젝트 진행 상황 점검
   - 백엔드 API 개발 90% 완료
   - 프론트엔드 UI 구현 70% 완료
   - 테스트 케이스 작성 60% 완료

2. 이슈 및 해결 방안
   - 데이터베이스 성능 최적화 필요
   - 사용자 인증 모듈 버그 수정 완료
   - 모바일 반응형 디자인 개선 진행 중

3. 다음 주 계획
   - 통합 테스트 시작
   - 사용자 피드백 수집
   - 배포 준비 작업

회의 종료 시간: 오후 3시 30분
"""
}

# 문서 요약 실행 

response = chain.invoke(
    input={
        "summary_style": "불릿포인트",
        "summary_length": "150",
        "document_type": "기술 문서",
        "document_title": "Langfuse 플랫폼 소개",
        "document_content": sample_documents["technical"]
    },
    config={"callbacks": [langfuse_handler]}
)

print("요약 결과:")
print(response.content)

# 모든 데이터를 Langfuse로 전송
langfuse.flush()

```
</details>
